# IB to VWAP Backtest

This backtest model is the conservative version, meaning that only one position can be open at a time. A new setup can only be taken once the previous position has been closed. This reduces the overall risk and capital requirements compared with the non-conservative version.

## 1. Data Preparation

The data preparation and indicator setup follows the same methodology as the research notebook, including timezone conversion, New York cash session filtering, Initial Balance and VWAP calculations. For this initial backtest, approximately six months of NQ data were used. This data was manually selected from the NQ daily chart using a daily ADX below 25, indicating relatively weak trend strength and a more consolidating market environment.

## 2. Strategy Rules

- IB: 9:30–10:30 New York session
- Setup: Price breaks below the IB Low, then reclaims it
- Entry: IB Low reclaim
- Target: VWAP
- Stop: 20 points
- Position management: Only one position can be open at a time
- New setups: Ignored while a position is already open
- Stop loss is moved to breakeven once the price has moved 20 points below entry price


In [1]:
#ib retracement/extension based off vwap 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file_paths = [
    r"C:\Users\charl\Downloads\NQ 03-26.Last 09.12.25-04.02.26 low adx.txt",
    r"C:\Users\charl\Downloads\NQ 09-26.Last 25.06-25.07 low adx.txt",
    r"C:\Users\charl\Downloads\NQ 12-25.Last 28.08.25-06.10.25.txt",
    r"C:\Users\charl\Downloads\NQ 09-25.Last 06.05.25.25.07.25.txt",
    r"C:\Users\charl\Downloads\NQ 09-26.Last 12.08.26-27.08.26.txt",
    r"C:\Users\charl\Downloads\NQ 12-25.Last 27.10.25-18.11.25.txt"
    
]

all_low_results = []


def load_data(file_path):

    df = pd.read_csv(
        file_path,
        sep=";",
        names=["date/time", "last", "bid", "ask", "volume"],
        skiprows=1,
        dtype={
            "date/time": "string",
            "last": "float32",
            "bid": "float32",
            "ask": "float32",
            "volume": "int32"
        }
    )

    df["date/time"] = pd.to_datetime(
        df["date/time"],
        format="%Y%m%d %H%M%S %f"
    )

    df = df.sort_values("date/time").reset_index(drop=True)

    return df

In [2]:
#converting from london timezone into New York using pythons standard IANA timezone database
def convert_timezone(df):

    df["date/time"] = (
        df["date/time"]
        .dt.tz_localize("Europe/London")
        .dt.tz_convert("America/New_York")
    )

    return df

In [3]:
#builidng a new dataframe for rth hours 
def create_rth(df):

    rth = df[
        (df["date/time"].dt.time >= pd.Timestamp("09:30").time()) &
        (df["date/time"].dt.time < pd.Timestamp("16:00").time())
    ].copy()

    return rth


In [4]:
#calcualting the initial balance(ib) for each day to be able to check with the live updating ib calculation that will be used in backtest
def calculate_ib_daily(rth):

    ib = rth[
        (rth["date/time"].dt.time >= pd.Timestamp("09:30").time()) &
        (rth["date/time"].dt.time < pd.Timestamp("10:30").time())
    ].copy()

    ib_daily = (
        ib.groupby(ib["date/time"].dt.date)
          .agg(
              ib_high_daily=("last", "max"),
              ib_low_daily=("last", "min")
          )
    )

    ib_daily["ib_range_daily"] = (
        ib_daily["ib_high_daily"] - ib_daily["ib_low_daily"]
    )

    ib_daily = ib_daily.reset_index()

    return ib_daily

In [5]:
#calculating the live ib values that will be used in backtest
def calculate_final_ib(rth):

    ib_live = (
        (rth["date/time"].dt.time >= pd.Timestamp("09:30").time()) &
        (rth["date/time"].dt.time < pd.Timestamp("10:30").time())
    )

    rth["ib_high"] = (
        rth["last"]
        .where(ib_live)
        .groupby(rth["date/time"].dt.date)
        .cummax()
    )

    rth["ib_low"] = (
        rth["last"]
        .where(ib_live)
        .groupby(rth["date/time"].dt.date)
        .cummin()
    )

    final_ib = (
        rth[ib_live]
        .groupby(rth.loc[ib_live, "date/time"].dt.date)[
            ["ib_high", "ib_low"]
        ]
        .last()
    )

    final_ib["ib_range"] = (
        final_ib["ib_high"] - final_ib["ib_low"]
    )

    return rth, final_ib

In [6]:
#making the ib only avaliable to use after 10:30 for the current rth session
def apply_final_ib(rth, final_ib):

    rth["ib_set"] = (
        rth["date/time"].dt.time >= pd.Timestamp("10:30").time()
    )

    rth["date"] = rth["date/time"].dt.date

    rth = rth.merge(
        final_ib,
        left_on="date",
        right_index=True,
        how="left",
        suffixes=("", "_final")
    )

    return rth

In [7]:
#building the VWAP for the rth session to use along side ib
#VWAP=∑(Price×Volume)/∑(Volume)
def calculate_vwap(rth):

    rth["price_volume"] = rth["last"] * rth["volume"]

    rth["cum_pv"] = (
        rth["price_volume"]
        .groupby(rth["date/time"].dt.date)
        .cumsum()
    )

    rth["cum_volume"] = (
        rth["volume"]
        .groupby(rth["date/time"].dt.date)
        .cumsum()
    )

    rth["vwap"] = (
        (rth["cum_pv"] / rth["cum_volume"]) / 0.25
    ).round() * 0.25

    return rth

In [8]:
# Identify IB Low break and reclaim event
def calculate_ib_events(rth):

    rth["below_ib_low"] = (
        rth["last"] < rth["ib_low_final"]
    )

    rth["had_ib_low_break"] = (
        rth["below_ib_low"]
        .groupby(rth["date/time"].dt.date)
        .cummax()
    )

    rth["ib_low_event"] = (
        rth["ib_set"] &
        rth["had_ib_low_break"] &
        (rth["last"].shift(1) < rth["ib_low_final"]) &
        (rth["last"] >= rth["ib_low_final"])
    )

    return rth

In [9]:
#backtest of the conservative model

def calculate_low_results(rth, stop=20, breakeven_trigger=20):

    low_results = []

    for date, day in rth.groupby(rth["date/time"].dt.date):

        events = day[day["ib_low_event"]]

        event_indices = events.index.tolist()

        i = 0

        while i < len(event_indices):

            idx = event_indices[i]

            event_position = day.index.get_loc(idx)
            future = day.iloc[event_position + 1:]

            if future.empty:
                break

            entry = day.loc[idx, "ib_low_final"]

            initial_stop = entry - stop
            breakeven_price = entry + breakeven_trigger

            breakeven_active = False

            exit_found = False

            for future_position, (future_idx, row) in enumerate(
                future.iterrows()
            ):

                price = row["last"]

                # Activate breakeven stop loss

                if not breakeven_active and price >= breakeven_price:
                    breakeven_active = True

                # Check stop

                if breakeven_active:

                    if price <= entry:

                        path = future.iloc[:future_position + 1]

                        mae = entry - path["last"].min()

                        low_results.append({
                            "date": date,
                            "event_time": day.loc[idx, "date/time"],
                            "entry": entry,
                            "exit": entry,
                            "exit_time": row["date/time"],
                            "result": "breakeven",
                            "vwap_hit": False,
                            "mae_to_vwap": mae,
                            "pnl": 0
                        })

                        exit_found = True
                        exit_position = future_position
                        break

                else:

                    if price <= initial_stop:

                        path = future.iloc[:future_position + 1]

                        mae = entry - path["last"].min()

                        low_results.append({
                            "date": date,
                            "event_time": day.loc[idx, "date/time"],
                            "entry": entry,
                            "exit": initial_stop,
                            "exit_time": row["date/time"],
                            "result": "loss",
                            "vwap_hit": False,
                            "mae_to_vwap": mae,
                            "pnl": -stop
                        })

                        exit_found = True
                        exit_position = future_position
                        break

                # Check VWAP

                if price >= row["vwap"]:

                    path = future.iloc[:future_position + 1]

                    mae = entry - path["last"].min()

                    low_results.append({
                        "date": date,
                        "event_time": day.loc[idx, "date/time"],
                        "entry": entry,
                        "exit": row["vwap"],
                        "exit_time": row["date/time"],
                        "result": "win",
                        "vwap_hit": True,
                        "mae_to_vwap": mae,
                        "pnl": row["vwap"] - entry
                    })

                    exit_found = True
                    exit_position = future_position
                    break

            if not exit_found:
                break

            # VWAP reached so stop trading for that day

            if low_results[-1]["result"] in ["win", "breakeven"]:
                break

            # Stop hit so look for next event

            next_event = None

            for j in range(i + 1, len(event_indices)):

                candidate_idx = event_indices[j]

                candidate_position = day.index.get_loc(candidate_idx)

                if candidate_position > event_position + exit_position:
                    next_event = j
                    break

            if next_event is None:
                break

            i = next_event

    return pd.DataFrame(low_results)

In [10]:
all_low_results = []

for file_path in file_paths:

    print(f"Processing: {file_path}")

    df = load_data(file_path)
    df = convert_timezone(df)

    rth = create_rth(df)

    rth, final_ib = calculate_final_ib(rth)
    rth = apply_final_ib(rth, final_ib)

    rth = calculate_vwap(rth)
    rth = calculate_ib_events(rth)

    low_results = calculate_low_results(rth, stop=20)

    all_low_results.append(low_results)

all_low_results = pd.concat(all_low_results, ignore_index=True)

Processing: C:\Users\charl\Downloads\NQ 03-26.Last 09.12.25-04.02.26 low adx.txt
Processing: C:\Users\charl\Downloads\NQ 09-26.Last 25.06-25.07 low adx.txt
Processing: C:\Users\charl\Downloads\NQ 12-25.Last 28.08.25-06.10.25.txt
Processing: C:\Users\charl\Downloads\NQ 09-25.Last 06.05.25.25.07.25.txt
Processing: C:\Users\charl\Downloads\NQ 09-26.Last 12.08.26-27.08.26.txt
Processing: C:\Users\charl\Downloads\NQ 12-25.Last 27.10.25-18.11.25.txt


In [11]:
# Trade analysis for the conservative model

trades = all_low_results.copy()

POINT_VALUE = 20

# Transaction costs
SLIPPAGE_PER_SIDE = 0.5
COMMISSION_PER_SIDE = 2.50

slippage_points = SLIPPAGE_PER_SIDE * 2
commission_dollars = COMMISSION_PER_SIDE * 2
commission_points = commission_dollars / POINT_VALUE

total_cost = slippage_points + commission_points

print("Cost per trade:")
print(f"Slippage:   {slippage_points:.2f} points")
print(f"Commission: ${commission_dollars:.2f}")
print(f"Total cost: {total_cost:.2f} points")


# Net P&L

trades["cost_points"] = total_cost
trades["net_pnl"] = trades["pnl"] - trades["cost_points"]

trades["gross_pnl_dollars"] = trades["pnl"] * POINT_VALUE
trades["net_pnl_dollars"] = trades["net_pnl"] * POINT_VALUE


# Performance

gross_pnl = trades["pnl"].sum()
net_pnl = trades["net_pnl"].sum()

gross_dollars = trades["gross_pnl_dollars"].sum()
net_dollars = trades["net_pnl_dollars"].sum()

wins = trades[trades["net_pnl"] > 0]
losses = trades[trades["net_pnl"] < 0]

win_rate = len(wins) / len(trades)

average_win = wins["net_pnl"].mean()
average_loss = losses["net_pnl"].mean()

expectancy = trades["net_pnl"].mean()
median_trade = trades["net_pnl"].median()

profit_factor = (
    wins["net_pnl"].sum()
    / abs(losses["net_pnl"].sum())
)


# Trading frequency

trading_days = trades["event_time"].dt.date.nunique()
trades_per_day = len(trades) / trading_days


# Position exposure

events = []

for _, trade in trades.iterrows():

    events.append({
        "time": trade["event_time"],
        "pnl_change": 0,
        "position_change": 1
    })

    events.append({
        "time": trade["exit_time"],
        "pnl_change": trade["net_pnl"],
        "position_change": -1
    })

events = pd.DataFrame(events)

# Exit before entry if timestamps are identical
events = events.sort_values(
    ["time", "position_change"]
).reset_index(drop=True)

events["open_positions"] = events["position_change"].cumsum()

max_positions = events["open_positions"].max()
average_positions = events["open_positions"].mean()


# Drawdown

events["cumulative_pnl"] = events["pnl_change"].cumsum()

events["running_peak"] = events["cumulative_pnl"].cummax()

events["drawdown"] = (
    events["cumulative_pnl"]
    - events["running_peak"]
)

max_drawdown = events["drawdown"].min()


# Daily P&L

events["date"] = events["time"].dt.date

daily_pnl = (
    events
    .groupby("date")["pnl_change"]
    .sum()
)

worst_day = daily_pnl.min()
best_day = daily_pnl.max()


# Daily Sharpe

daily_sharpe = (
    daily_pnl.mean()
    / daily_pnl.std()
) * np.sqrt(252)


# Dollar values

max_drawdown_dollars = max_drawdown * POINT_VALUE
worst_day_dollars = worst_day * POINT_VALUE
best_day_dollars = best_day * POINT_VALUE


# Results

print("\nNET PERFORMANCE AFTER TRANSACTION COSTS")
print("----------------------------------------")

print(f"Trades:              {len(trades)}")
print(f"Trading days:        {trading_days}")
print(f"Trades per day:      {trades_per_day:.2f}")

print(f"Maximum positions:   {max_positions}")
print(f"Average positions:   {average_positions:.2f}")

print(f"\nMaximum drawdown:     {max_drawdown:.2f} points")
print(f"Maximum drawdown:     ${max_drawdown_dollars:,.2f}")

print(f"\nWorst day:            {worst_day:.2f} points")
print(f"Worst day:            ${worst_day_dollars:,.2f}")

print(f"\nBest day:             {best_day:.2f} points")
print(f"Best day:             ${best_day_dollars:,.2f}")

print(f"\nDaily Sharpe:         {daily_sharpe:.2f}")

print(f"\nGross P&L:            {gross_pnl:.2f} points")
print(f"Transaction costs:    {gross_pnl - net_pnl:.2f} points")
print(f"Net P&L:              {net_pnl:.2f} points")

print(f"\nGross P&L:            ${gross_dollars:,.2f}")
print(f"Net P&L:              ${net_dollars:,.2f}")

print(f"\nWin rate:             {win_rate:.2%}")
print(f"Average win:          {average_win:.2f} points")
print(f"Average loss:         {average_loss:.2f} points")
print(f"Expectancy:           {expectancy:.2f} points")
print(f"Median trade:         {median_trade:.2f} points")
print(f"Profit factor:        {profit_factor:.2f}")

Cost per trade:
Slippage:   1.00 points
Commission: $5.00
Total cost: 1.25 points

NET PERFORMANCE AFTER TRANSACTION COSTS
----------------------------------------
Trades:              131
Trading days:        77
Trades per day:      1.70
Maximum positions:   1
Average positions:   0.48

Maximum drawdown:     -380.00 points
Maximum drawdown:     $-7,600.00

Worst day:            -106.25 points
Worst day:            $-2,125.00

Best day:             131.00 points
Best day:             $2,620.00

Daily Sharpe:         0.96

Gross P&L:            360.25 points
Transaction costs:    163.75 points
Net P&L:              196.50 points

Gross P&L:            $7,205.00
Net P&L:              $3,930.00

Win rate:             27.48%
Average win:          44.49 points
Average loss:         -14.79 points
Expectancy:           1.50 points
Median trade:         -1.25 points
Profit factor:        1.14
